# Inspect WOLF Data

Load WOLF data to investigate why it shows high momentum despite only 3 months of trading.

In [1]:
import sqlite3
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

In [2]:
# Connect to database
db_file = Path('data/trading.db')
conn = sqlite3.connect(db_file)
print(f"Connected to {db_file}")

Connected to data/trading.db


In [3]:
# Get WOLF's full trading history
query = """
SELECT 
    date,
    open,
    high,
    low,
    close,
    volume
FROM daily_prices
WHERE ticker = 'WOLF'
ORDER BY date
"""

df_wolf = pd.read_sql_query(query, conn)
df_wolf['date'] = pd.to_datetime(df_wolf['date'])

print(f"WOLF trading history:")
print(f"  First date: {df_wolf['date'].min().date()}")
print(f"  Last date: {df_wolf['date'].max().date()}")
print(f"  Trading days: {len(df_wolf)}")
print(f"  Calendar days: {(df_wolf['date'].max() - df_wolf['date'].min()).days}")

df_wolf

WOLF trading history:
  First date: 2021-10-04
  Last date: 2025-12-04
  Trading days: 1048
  Calendar days: 1522


,date,open,high,low,close,volume
0,2021-10-04,83.2500,85.00,79.3300,80.07,860982
1,2021-10-05,80.9900,83.16,79.0100,82.88,1375826
2,2021-10-06,81.4000,83.60,80.2700,82.95,1035235
3,2021-10-07,84.6500,85.67,83.8400,84.27,1151935
4,2021-10-08,86.3000,86.30,83.4800,83.66,791347
...,...,...,...,...,...,...
1043,2025-11-28,20.4000,20.86,19.9100,20.28,420327
1044,2025-12-01,21.8937,23.12,20.5001,22.11,3684990
1045,2025-12-02,21.7900,22.30,20.5000,21.36,1829032
1046,2025-12-03,21.1100,21.88,20.7100,21.87,1060428


In [4]:
# Get WOLF with split adjustments and momentum data
query_momentum = """
SELECT 
    mc.date,
    ap.adj_close,
    ap.adj_volume,
    mc.price_lag126,
    mc.date_lag126,
    mc.days_since_lag,
    mc.momentum_6m,
    ef.avg_price_6m,
    ef.avg_dollar_volume_1m,
    ef.is_eligible,
    mr.momentum_rank
FROM momentum_calc mc
JOIN adjusted_prices ap ON mc.ticker = ap.ticker AND mc.date = ap.date
JOIN eligibility_filters ef ON mc.ticker = ef.ticker AND mc.date = ef.date
JOIN momentum_rankings mr ON mc.ticker = mr.ticker AND mc.date = mr.date
WHERE mc.ticker = 'WOLF'
ORDER BY mc.date
"""

df_wolf_momentum = pd.read_sql_query(query_momentum, conn)
df_wolf_momentum['date'] = pd.to_datetime(df_wolf_momentum['date'])

print("\nWOLF with momentum calculations:")
df_wolf_momentum


WOLF with momentum calculations:


,date,adj_close,adj_volume,price_lag126,date_lag126,days_since_lag,momentum_6m,avg_price_6m,avg_dollar_volume_1m,is_eligible,momentum_rank
0,2021-10-04,80.07,860982,NaN,None,NaN,NaN,NaN,NaN,0,NaN
1,2021-10-05,82.88,1375826,NaN,None,NaN,NaN,NaN,NaN,0,NaN
2,2021-10-06,82.95,1035235,NaN,None,NaN,NaN,NaN,NaN,0,NaN
3,2021-10-07,84.27,1151935,NaN,None,NaN,NaN,NaN,NaN,0,NaN
4,2021-10-08,83.66,791347,NaN,None,NaN,NaN,NaN,NaN,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1043,2025-11-28,20.28,420327,1.19,2025-05-30,182.0,16.042017,9.607194,3.993532e+07,1,7.0
1044,2025-12-01,22.11,3684990,1.21,2025-06-02,182.0,17.272727,9.773067,3.623452e+07,1,6.0
1045,2025-12-02,21.36,1829032,1.40,2025-06-03,182.0,14.257143,9.931479,3.514177e+07,1,7.0
1046,2025-12-03,21.87,1060428,1.61,2025-06-04,182.0,12.583851,10.092273,3.422442e+07,1,9.0


In [ ]:
# Check for splits
splits_query = """
SELECT *
FROM splits
WHERE ticker = 'WOLF'
"""

df_splits = pd.read_sql_query(splits_query, conn)
print("WOLF Splits:")
if len(df_splits) > 0:
    df_splits
else:
    print("No splits found")

In [ ]:
# Show recent data with all details
print("\nWOLF recent data (last 20 rows):")
df_wolf_momentum.tail(20)[[
    'date', 'adj_close', 'date_lag126', 'price_lag126', 
    'days_since_lag', 'momentum_6m', 'is_eligible', 'momentum_rank'
]]

In [ ]:
# Chart WOLF price history
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_wolf_momentum['date'],
    y=df_wolf_momentum['adj_close'],
    mode='lines+markers',
    name='Adjusted Close',
    line=dict(color='blue', width=2)
))

fig.update_layout(
    title='WOLF - Price History',
    xaxis_title='Date',
    yaxis_title='Adjusted Close Price ($)',
    height=600,
    hovermode='x unified'
)

fig.show()

In [ ]:
# Analyze the momentum calculation for latest date
latest = df_wolf_momentum.iloc[-1]

print("\n" + "="*70)
print("WOLF Latest Data Analysis")
print("="*70)
print(f"Date: {latest['date'].date()}")
print(f"Current adj_close: ${latest['adj_close']:.2f}")
print(f"Date 126 days ago: {latest['date_lag126']}")
print(f"Price 126 days ago: ${latest['price_lag126']:.2f}" if pd.notna(latest['price_lag126']) else "Price 126 days ago: N/A")
print(f"Days since lag: {latest['days_since_lag']:.0f}" if pd.notna(latest['days_since_lag']) else "Days since lag: N/A")
print(f"Momentum: {latest['momentum_6m']:.2%}" if pd.notna(latest['momentum_6m']) else "Momentum: N/A")
print(f"Is eligible: {latest['is_eligible'] == 1}")
print(f"Momentum rank: {latest['momentum_rank']:.0f}" if pd.notna(latest['momentum_rank']) else "Momentum rank: N/A")
print(f"\nAvg 6M price: ${latest['avg_price_6m']:.2f}" if pd.notna(latest['avg_price_6m']) else "\nAvg 6M price: N/A")
print(f"Avg monthly $ volume: ${latest['avg_dollar_volume_1m']:,.0f}" if pd.notna(latest['avg_dollar_volume_1m']) else "Avg monthly $ volume: N/A")

In [ ]:
# Check if WOLF has less than 126 trading days total
print(f"\nDiagnostic:")
print(f"Total WOLF trading days: {len(df_wolf)}")
if len(df_wolf) < 126:
    print(f"⚠ WOLF has only {len(df_wolf)} trading days, but 126 are needed for momentum!")
    print(f"This means the LAG(126) function returns data from BEFORE WOLF started trading.")
    print(f"This could be picking up data from a different ticker or merger/rebranding.")
else:
    print(f"✓ WOLF has {len(df_wolf)} trading days (>= 126 needed)")

In [ ]:
# Close connection
conn.close()
print("\nDatabase connection closed")